# 04 - Transformer (D)

In [1]:
import sys
sys.path.append("..")

#from src.config import *
#from src.data import load_split
#from src.cleaning import clean_text
#from src.evaluate import evaluate, save_results

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

x = torch.rand(1000, 1000, device="cuda")
y = x @ x
print("Result:", y.sum().item())

PyTorch version: 2.15.0+cu132
CUDA available: True
GPU name: NVIDIA GeForce RTX 5060 Ti
Result: 250409344.0


In [3]:
import pandas as pd

DATA_DIR = "../data/raw"

def load_raw_split(split):
    """Read one split ("train", "val" or "test") of TweetEval irony into a DataFrame."""
    with open(f"{DATA_DIR}/{split}_text.txt", encoding="utf-8") as f:
        texts = f.read().splitlines()

    with open(f"{DATA_DIR}/{split}_labels.txt", encoding="utf-8") as f:
        labels = [int(line) for line in f.read().splitlines()]

    assert len(texts) == len(labels), f"{split}: {len(texts)} texts but {len(labels)} labels"

    return pd.DataFrame({"text": texts, "label": labels})


train_df = load_raw_split("train")
val_df = load_raw_split("val")
test_df = load_raw_split("test")

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

train: (2862, 2)
val:   (955, 2)
test:  (784, 2)


In [4]:
train_df.head()

,text,label
0,seeing ppl walking w/ crutches makes me really...,1
1,"look for the girl with the broken smile, ask h...",0
2,Now I remember why I buy books online @user #s...,1
3,@user @user So is he banded from wearing the c...,1
4,Just found out there are Etch A Sketch apps. ...,1


In [5]:
corrupted = train_df["text"].str.contains("\t")

print("Corrupted rows found:", corrupted.sum())
print(train_df.loc[corrupted, "text"].str.len())

train_df = train_df[~corrupted].reset_index(drop=True)
print("train after removal:", train_df.shape)

Corrupted rows found: 3
1295    1000
1753     329
2773     249
Name: text, dtype: int64
train after removal: (2859, 2)


In [6]:
from datasets import Dataset, DatasetDict

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2859
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 955
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 784
    })
})
{'text': 'seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of my life  ', 'label': 1}


In [7]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = ds["train"][0]["text"]
encoded = tokenizer(example)

print("Tweet:    ", example)
print("input_ids:", encoded["input_ids"])
print("tokens:   ", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("attention:", encoded["attention_mask"])

Tweet:     seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of my life  
input_ids: [101, 3773, 4903, 2140, 3788, 1059, 1013, 13675, 4904, 8376, 3084, 2033, 2428, 7568, 2005, 1996, 2279, 1017, 3134, 1997, 2026, 2166, 102]
tokens:    ['[CLS]', 'seeing', 'pp', '##l', 'walking', 'w', '/', 'cr', '##ut', '##ches', 'makes', 'me', 'really', 'excited', 'for', 'the', 'next', '3', 'weeks', 'of', 'my', 'life', '[SEP]']
attention: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [8]:
roberta_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

test_tweet = "Oh GREAT, another Monday 🙂"

for name, tok in [("DistilBERT", tokenizer), ("RoBERTa", roberta_tokenizer)]:
    ids = tok(test_tweet)["input_ids"]
    print(f"{name:10} -> {tok.convert_ids_to_tokens(ids)}")

DistilBERT -> ['[CLS]', 'oh', 'great', ',', 'another', 'monday', '[UNK]', '[SEP]']
RoBERTa    -> ['<s>', 'Oh', 'ĠGREAT', ',', 'Ġanother', 'ĠMonday', 'ĠðŁĻĤ', '</s>']


In [9]:
token_lengths = [len(tokenizer(text)["input_ids"]) for text in train_df["text"]]

lengths = pd.Series(token_lengths)
print(lengths.describe())
print("99th percentile:", lengths.quantile(0.99))

count    2859.000000
mean       22.730325
std         9.082953
min         3.000000
25%        15.000000
50%        22.000000
75%        30.000000
max        60.000000
dtype: float64
99th percentile: 44.0


In [10]:
print(train_df.loc[lengths.idxmax(), "text"])

@user Hey #Russia ! MAKE MONEY #TWEETING YET?||http://t.co/Q2WB7riAvK <-This #FOXNews Clip explains how||https://t.co/cpJZkEgNe7 


In [11]:
MAX_LENGTH = 64

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_ds = ds.map(tokenize, batched=True)

print(tokenized_ds)
print(tokenized_ds["train"][0])

Map:   0%|          | 0/2859 [00:00<?, ? examples/s]

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

Map:   0%|          | 0/784 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2859
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 955
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 784
    })
})
{'text': 'seeing ppl walking w/ crutches makes me really excited for the next 3 weeks of my life  ', 'label': 1, 'input_ids': [101, 3773, 4903, 2140, 3788, 1059, 1013, 13675, 4904, 8376, 3084, 2033, 2428, 7568, 2005, 1996, 2279, 1017, 3134, 1997, 2026, 2166, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [12]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "non_irony", 1: "irony"}
label2id = {"non_irony": 0, "irony": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

print(model.classifier)
print(f"Parameters: {model.num_parameters():,}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)
Parameters: 66,955,010


In [13]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }

In [15]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/distilbert-irony",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    fp16=True,
    seed=42,
    report_to="none",
)

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.645253,0.613803,0.652356,0.646492
2,0.522039,0.628649,0.667016,0.666459
3,0.387162,0.647394,0.685864,0.685533
4,0.287587,0.707218,0.681675,0.681672


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=716, training_loss=0.46051007276140776, metrics={'train_runtime': 35.8977, 'train_samples_per_second': 318.572, 'train_steps_per_second': 19.946, 'total_flos': 115405235493564.0, 'train_loss': 0.46051007276140776, 'epoch': 4.0})

In [17]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.287587,0.647394,4,0.685864,0.685533


{'eval_loss': 0.6473937630653381,
 'eval_accuracy': 0.6858638743455497,
 'eval_f1_macro': 0.6855325202181186}